# TTA text-only timeline vs. the owner's weaning timestamps

Compares the TTA **text-only** run (`unimodal/`, qwen3.8-27b via the local
proxy) on the 347 shift summaries against the owner's curated event
timestamps (`table6_weaning.csv` for now).

**Pipeline**

1. Load the reference truth (long format) and the exported note manifest.
2. Load the TTA run's `timeline.json` files (`[event, hours]` pairs).
3. **Anchor** each note's hour axis: match TTA events to timed lines (from the
   owner's clock-resolved `event_nodes` when available, else the note's own
   clock lines) and take the median of `line_time - hours` as the origin,
   so `abs_time = origin + hours`.
4. Match TTA events to the target concepts (admission / separation attempt /
   weaning) by keyword, review the candidates, and pick the earliest absolute
   time per (patient, event) — with manual `OVERRIDES` where needed.
5. Compute the difference vs. the reference in minutes (definitions follow
   `weaning_event/time_difference_findings.md`: `diff = TTA - reference`).

**Prerequisites**

* Notes already exported: `comparison/data/` (see `../README.md`).
* A TTA run already finished: `bash comparison/run_text_only.sh --all`
  (results under `comparison/outputs/<run>/note_<id>/timeline.json`).

Every threshold and the event mapping is an editable variable in the
**Parameters** cell below.

**Outputs** (when `WRITE_OUTPUTS = True`): `comparison/out/*.csv`, including
`all_event_decisions_tta.csv` in the owner's `all_event_decisions` schema.

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd

# Repo root = the directory containing unimodal/ (works from any cwd).
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "unimodal").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from comparison.compare import anchors as anchorlib
from comparison.compare import loaders, metrics, targets

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 100)
print("repo root:", ROOT)

## Parameters

Note: `table6_weaning.csv` has **no** `visit_start` row, so the
`admission → visit_start` mapping shows `n_ref = 0` against it. `visit_start`
is available in `weaning_structured.csv` (set `REFERENCE_KIND = "wide"` to
compare admission too).

In [ ]:
DATA_DIR = ROOT / "comparison" / "data"
OUTPUTS_DIR = ROOT / "comparison" / "outputs"
OUT_DIR = ROOT / "comparison" / "out"

# Owner's reference truth. Long format: patient_id, source, event, time.
REFERENCE_PATH = Path(
    os.environ.get(
        "TTA_REFERENCE_CSV",
        "/mnt/c/Users/Karen/Documents/GitHub/LLMs_for_feature_extration_test/"
        "physiological_events/tables/table6_weaning.csv",
    )
)
REFERENCE_KIND = "long"          # "long" (table6_weaning.csv) or "wide" (weaning_structured.csv)
REFERENCE_SOURCE = "structured"  # which `source` rows to compare against

# Optional clock-resolved event nodes (preferred anchor source).
NODES_PATH = Path(
    os.environ.get(
        "TTA_NODES_JSONL",
        "/mnt/c/Users/Karen/Documents/GitHub/LLMs_for_feature_extration_test/"
        "event_nodes/results/agents/agent_qwen38_27b_vllm/run-1/nodes.jsonl",
    )
)

# TTA run to compare; "" picks the newest directory under outputs/.
RUN_NAME = ""

# TTA concept -> reference event name.
EVENT_MAP = {
    "admission": "visit_start",
    "separation_attempt": "first_low_support_time",
    "weaning": "first_weaning",
}

# Extra reference: owner's LLM-evaluation answers (investigate_extubation).
# Wide file, ISO timestamps, "Non present" for absent values.
EXTRAP_PATH = Path(
    os.environ.get(
        "TTA_EXTRAP_CSV",
        "/mnt/c/Users/Karen/Documents/GitHub/LLMs_for_feature_extration_test/"
        "Evluating_llms/results/investigate_extubation_output/all_answers_33.csv",
    )
)
# TTA concept -> all_answers_33.csv column.
EXTRAP_EVENT_MAP = {
    "separation_attempt": "first_imv_end",
    "weaning": "first_weaning",
    "extubation_plan": "extubation_plan",
}

# Matching / anchoring thresholds.
# 0.7 keeps substring/paraphrase matches (score 1.0) and drops generic
# clinical phrasings that char-level similarity picks up (e.g. "Plan for
# CVC line" vs "plan for extubation", ~0.55). Lower it to review more.
MATCH_THRESHOLD = 0.7       # min concept score for a TTA event to be a candidate
ANCHOR_SOURCE = "event_nodes"  # "event_nodes" (preferred) or "note_lines"
MIN_PAIRS = 3               # matched pairs needed to trust an anchor estimate
MIN_SCORE = 0.35            # similarity needed for an anchor pair
MAX_MAD_MIN = 120.0         # residual spread (min) for an anchor to count as stable

WRITE_OUTPUTS = True        # write CSVs to comparison/out/

## 1. Reference truth

Long table: one row per (patient, event) with a curated timestamp. Restricted
to the target events and the selected `source`.

In [ ]:
if not REFERENCE_PATH.is_file():
    raise FileNotFoundError(
        f"reference not found: {REFERENCE_PATH} (edit REFERENCE_PATH or set TTA_REFERENCE_CSV)"
    )

if REFERENCE_KIND == "wide":
    reference = loaders.load_structured_wide(REFERENCE_PATH, events=list(EVENT_MAP.values()))
else:
    reference = loaders.load_long_reference(REFERENCE_PATH)

reference = (
    reference[reference["event"].isin(EVENT_MAP.values())]
    .drop_duplicates(subset=["patient_id", "event"], keep="first")
    .reset_index(drop=True)
)
if REFERENCE_SOURCE:
    reference = reference[reference["source"] == REFERENCE_SOURCE].reset_index(drop=True)

patients = sorted(reference["patient_id"].dropna().unique())
print(f"{len(patients)} patients, {len(reference)} target timestamps")
display(reference.pivot_table(index="patient_id", columns="event", values="time", aggfunc="first"))

## 2. Notes and TTA run

The manifest links each note to its patient and entry time (`note_datetime`);
the TTA run is discovered under `comparison/outputs/`.

In [ ]:
notes = loaders.load_note_manifest(DATA_DIR / "manifest.csv")
print(f"{len(notes)} notes, {notes['person_id'].nunique()} patients")

runs = (
    sorted((p for p in OUTPUTS_DIR.iterdir() if p.is_dir()), key=lambda p: p.name)
    if OUTPUTS_DIR.exists()
    else []
)
run_dir = OUTPUTS_DIR / RUN_NAME if RUN_NAME else (runs[-1] if runs else None)
if run_dir is None or not run_dir.is_dir():
    raise FileNotFoundError(
        f"no TTA run under {OUTPUTS_DIR}; run `bash comparison/run_text_only.sh --all` first"
    )
print("comparing run:", run_dir)

tta = loaders.load_tta_run(run_dir)
print(f"{len(tta)} predicted events across {tta['note_id'].nunique() if len(tta) else 0} notes")
if tta.empty:
    print("Nothing to compare yet - the run produced no timeline events.")

## 3. Anchoring the hour axis

TTA emits `event|hours` relative to its own time zero. Per note, each TTA
event is matched to a timed line (owner's clock-resolved `event_nodes` first,
the note's own clock lines otherwise); the **median** of `line_time - hours`
is the note's origin, and `abs_time = origin + hours`. `mad_min` is the
residual spread; unstable notes are flagged, not silently trusted.

In [ ]:
import matplotlib.pyplot as plt

nodes = None
if not tta.empty and NODES_PATH.is_file():
    nodes = loaders.load_event_nodes(NODES_PATH, note_ids=tta["note_id"].unique())
    print(f"event_nodes anchor candidates: {len(nodes)} rows")
else:
    print("event_nodes file not available - anchoring from note clock lines")

anchors = anchorlib.estimate_all_anchors(
    notes,
    tta,
    DATA_DIR,
    nodes=nodes,
    anchor_source=ANCHOR_SOURCE,
    min_pairs=MIN_PAIRS,
    min_score=MIN_SCORE,
    max_mad_min=MAX_MAD_MIN,
)
print(metrics.anchor_summary(anchors).to_string(index=False) if len(anchors) else "no anchors (empty TTA run)")

if len(anchors):
    display(anchors.sort_values("stable").sort_values("mad_min", ascending=False).head(15))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
    anchors["offset_min"].hist(ax=ax1, bins=30, color="steelblue", edgecolor="white")
    ax1.set_title("anchor origin - note_datetime (minutes)")
    anchors["mad_min"].hist(ax=ax2, bins=30, color="seagreen", edgecolor="white")
    ax2.set_title("anchor residual MAD (minutes)")
    fig.tight_layout()

## 4. Predicted events with absolute times

In [ ]:
tta_abs = anchorlib.apply_anchors(tta, anchors)
tta_abs = tta_abs.merge(notes[["note_id", "person_id"]], on="note_id", how="left")
cols = ["note_id", "person_id", "event", "hours", "abs_time", "source", "stable", "n_pairs", "mad_min"]
display(tta_abs[cols].head(25) if len(tta_abs) else None)
print(f"{tta_abs['abs_time'].isna().sum()} events without an absolute time")

## 5. Candidate review

Every TTA event whose concept score is at least `MATCH_THRESHOLD`, joined to
the reference timestamp of the same event. Inspect this before step 7:
generic wording (e.g. `... commenced`) can score higher than expected.

In [ ]:
candidates = targets.candidate_table(tta_abs, threshold=MATCH_THRESHOLD)
if len(candidates):
    review = candidates.copy()
    review["ref_event"] = review["concept"].map(EVENT_MAP)
    review = review.merge(
        reference[["patient_id", "event", "time"]].rename(columns={"time": "ref_time"}),
        left_on=["patient_id", "ref_event"],
        right_on=["patient_id", "event"],
        how="left",
    )
    review["diff_from_ref_min"] = (review["abs_time"] - review["ref_time"]).dt.total_seconds() / 60.0
    review["abs_time"] = review["abs_time"].dt.strftime("%Y-%m-%d %H:%M")
    review["ref_time"] = review["ref_time"].dt.strftime("%Y-%m-%d %H:%M")
    display(review.sort_values(["patient_id", "concept", "abs_time"]).to_string(index=False))
else:
    print("no candidates - check MATCH_THRESHOLD and the TTA run above")

## 6. Manual overrides

Pin a specific candidate per (patient, concept) before the earliest absolute
time is picked. `note_id` and/or `contains` (substring of the TTA event text)
narrow the pool.

In [ ]:
OVERRIDES = {
    # (191731, "weaning"): {"note_id": 12407, "contains": "sbt"},
}

## 7. Resolve one timestamp per (patient, event)

The earliest absolute time among the (possibly overridden) candidates wins;
patients with no candidate get a `MISSING` row.

In [ ]:
resolved = targets.resolve_targets(candidates, EVENT_MAP, overrides=OVERRIDES, patients=patients)
display(resolved.to_string(index=False))
print(resolved["status"].value_counts().to_dict())

## 8. Comparison vs. the reference

`diff_min = TTA - reference` in minutes (signed; median and tolerance bands
as in `weaning_event/time_difference_findings.md`).

In [ ]:
comparison = metrics.build_comparison(
    resolved, reference, EVENT_MAP, source=REFERENCE_SOURCE or "structured"
)
comparison = comparison.copy()
comparison["source"] = REFERENCE_SOURCE or "structured"
summary = metrics.summarize(comparison)
display(summary.to_string(index=False))

show = comparison.copy()
for column in ("ref_time", "tta_time"):
    show[column] = show[column].dt.strftime("%Y-%m-%d %H:%M")
display(show.sort_values(["event", "abs_diff_min"], na_position="last").to_string(index=False))

In [ ]:
matched = comparison[comparison["matched"]]
if len(matched):
    events = sorted(matched["event"].unique())
    fig, axes = plt.subplots(len(events), 1, figsize=(8, 3.0 * len(events)), squeeze=False)
    for ax, event in zip(axes.flat, events):
        diff = matched.loc[matched["event"] == event, "diff_min"]
        ax.hist(diff, bins=20, color="steelblue", edgecolor="white", align="mid")
        ax.axvline(0.0, color="k", lw=0.8)
        ax.set_title(f"{event}: TTA - reference (minutes), n={len(diff)}")
        ax.set_xlabel("difference (minutes)")
    fig.tight_layout()
else:
    print("no matched pairs to plot")

## 9. Write outputs

In [ ]:
if WRITE_OUTPUTS:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    comparison.to_csv(OUT_DIR / "tta_vs_reference.csv", index=False)
    summary.to_csv(OUT_DIR / "tta_vs_reference_summary.csv", index=False)
    resolved.to_csv(OUT_DIR / "tta_targets.csv", index=False)
    anchors.to_csv(OUT_DIR / "tta_anchor_estimates.csv", index=False)
    decisions = metrics.to_event_decisions(resolved, run_id=run_dir.name if run_dir else "tta")
    decisions.to_csv(OUT_DIR / "all_event_decisions_tta.csv", index=False)
    print("wrote to", OUT_DIR)
else:
    print("WRITE_OUTPUTS is False - nothing written")

## 10. Comparison with the extubation-study answers (`all_answers_33.csv`)

Second reference: the owner's LLM-evaluation answers
(`Evluating_llms/results/investigate_extubation_output/all_answers_33.csv`,
15 patients). Timestamps are ISO and absent values are the literal string
`"Non present"`, so it is parsed with an explicit `ISO8601` format (never
`dayfirst`, which corrupts ISO dates).

Mapping (TTA concept -> answer column):

* `separation_attempt -> first_imv_end` (end of invasive ventilation)
* `weaning -> first_weaning`
* `extubation_plan -> extubation_plan` (first documented wean/extubation
  plan; past-tense and "not for extubation" phrasings are penalised in
  `targets.CONCEPTS["extubation_plan"]`)

Caveats: `first_imv_end` is not strictly `first_low_support_time` —
`separation_attempt` is the closest TTA concept. Reuses the candidates from
section 5 and the `OVERRIDES` from section 6; same `diff = TTA - reference`
definitions as before.

The write step below **combines both comparisons** into the final
`tta_vs_reference.csv` and `tta_vs_reference_summary.csv`: one block per
`source` (`structured` from section 8, `chained` from here).

In [ ]:
if not EXTRAP_PATH.is_file():
    raise FileNotFoundError(
        f"extra reference not found: {EXTRAP_PATH} (edit EXTRAP_PATH or set TTA_EXTRAP_CSV)"
    )

extrap = loaders.load_structured_wide(
    EXTRAP_PATH,
    events=list(EXTRAP_EVENT_MAP.values()),
    patient_col="patient_id",
    format="ISO8601",
)
extrap = (
    extrap[extrap["event"].isin(EXTRAP_EVENT_MAP.values())]
    .drop_duplicates(subset=["patient_id", "event"], keep="first")
    .reset_index(drop=True)
)
patients_extrap = sorted(extrap["patient_id"].dropna().unique())
print(f"{len(patients_extrap)} patients, {len(extrap)} answer timestamps")
display(extrap.pivot_table(index="patient_id", columns="event", values="time", aggfunc="first"))

In [ ]:
resolved_extrap = targets.resolve_targets(
    candidates, EXTRAP_EVENT_MAP, overrides=OVERRIDES, patients=patients_extrap
)
comparison_extrap = metrics.build_comparison(
    resolved_extrap, extrap, EXTRAP_EVENT_MAP, source="structured"
)
comparison_extrap = comparison_extrap.copy()
comparison_extrap["source"] = "chained"
summary_extrap = metrics.summarize(comparison_extrap)
display(summary_extrap.to_string(index=False))

show_extrap = comparison_extrap.copy()
for column in ("ref_time", "tta_time"):
    show_extrap[column] = show_extrap[column].dt.strftime("%Y-%m-%d %H:%M")
display(show_extrap.sort_values(["event", "abs_diff_min"], na_position="last").to_string(index=False))

In [ ]:
matched_extrap = comparison_extrap[comparison_extrap["matched"]]
if len(matched_extrap):
    events_extrap = sorted(matched_extrap["event"].unique())
    fig, axes = plt.subplots(len(events_extrap), 1, figsize=(8, 3.0 * len(events_extrap)), squeeze=False)
    for ax, event in zip(axes.flat, events_extrap):
        diff = matched_extrap.loc[matched_extrap["event"] == event, "diff_min"]
        ax.hist(diff, bins=20, color="indianred", edgecolor="white", align="mid")
        ax.axvline(0.0, color="k", lw=0.8)
        ax.set_title(f"{event}: TTA - answers (minutes), n={len(diff)}")
        ax.set_xlabel("difference (minutes)")
    fig.tight_layout()
else:
    print("no matched pairs to plot")

In [ ]:
if WRITE_OUTPUTS:
    # Combined reference: structured (section 8) + chained (this section).
    combined = pd.concat([comparison, comparison_extrap], ignore_index=True)
    combined.to_csv(OUT_DIR / "tta_vs_reference.csv", index=False)
    summary_structured = summary.copy()
    summary_structured.insert(0, "source", REFERENCE_SOURCE or "structured")
    summary_chained = summary_extrap.copy()
    summary_chained.insert(0, "source", "chained")
    pd.concat([summary_structured, summary_chained], ignore_index=True).to_csv(
        OUT_DIR / "tta_vs_reference_summary.csv", index=False
    )
    comparison_extrap.to_csv(OUT_DIR / "tta_vs_extubation_answers.csv", index=False)
    summary_extrap.to_csv(OUT_DIR / "tta_vs_extubation_answers_summary.csv", index=False)
    decisions_extrap = metrics.to_event_decisions(
        resolved_extrap, run_id=run_dir.name if run_dir else "tta"
    )
    decisions_extrap.to_csv(OUT_DIR / "all_event_decisions_tta_extubation.csv", index=False)
    print("wrote combined tta_vs_reference.csv (+ extubation-answers files) to", OUT_DIR)
else:
    print("WRITE_OUTPUTS is False - nothing written")

## 11. Figure - time difference vs. the structured reference

One figure (in the style of `Evluating_llms/last_ML_run_report.ipynb`)
showing how far each system's timestamps sit from the curated `structured`
reference: median of `diff = system - structured`, matched (patient, event)
pairs only.

* **TTA** - this repository's text-only timelines (section 8)
* **chained** - the owner's chained LLM answers (`all_answers_33.csv`)
* **raw notes** - the `source = "notes"` rows of `table6_weaning.csv` (a
  separate chained extraction from the notes; long-format reference only)

Panels: signed and absolute medians, **per event** and **pooled** over each
system's available events; bars annotate the pair count `n`. Events without
a structured row in `table6` (`extubation_plan`, `visit_start`) cannot be
shown here. Saved to `comparison/out/tta_time_difference.png`.

In [ ]:
# Long table of matched time differences: diff = system - structured (minutes).
# TTA uses the section-8 comparison; chained uses all_answers_33.csv; raw notes
# use the source="notes" rows of table6 (long format only).
if REFERENCE_KIND == "long":
    ref_all = loaders.load_long_reference(REFERENCE_PATH)
    ref_struct = ref_all[ref_all["source"] == "structured"].drop_duplicates(subset=["patient_id", "event"], keep="first")
    notes_part = ref_all[ref_all["source"] == "notes"].drop_duplicates(subset=["patient_id", "event"], keep="first")
else:
    ref_struct = reference.drop_duplicates(subset=["patient_id", "event"], keep="first")
    notes_part = pd.DataFrame()

diff_rows = []
tta_part = comparison[comparison["matched"]]
diff_rows.append(tta_part[["patient_id", "event", "diff_min"]].assign(system="TTA"))

m = extrap.merge(ref_struct[["patient_id", "event", "time"]], on=["patient_id", "event"], how="inner", suffixes=("", "_ref"))
if len(m):
    diff_rows.append(pd.DataFrame({
        "patient_id": m["patient_id"], "event": m["event"],
        "diff_min": (m["time"] - m["time_ref"]).dt.total_seconds() / 60.0,
    }).assign(system="chained"))

if len(notes_part):
    m2 = notes_part.merge(ref_struct[["patient_id", "event", "time"]], on=["patient_id", "event"], how="inner", suffixes=("", "_ref"))
    if len(m2):
        diff_rows.append(pd.DataFrame({
            "patient_id": m2["patient_id"], "event": m2["event"],
            "diff_min": (m2["time"] - m2["time_ref"]).dt.total_seconds() / 60.0,
        }).assign(system="raw notes"))

diff_long = pd.concat(diff_rows, ignore_index=True)
diff_long["abs_diff_min"] = diff_long["diff_min"].abs()
display(diff_long.groupby(["system", "event"]).agg(
    n=("diff_min", "size"),
    median_min=("diff_min", "median"),
    median_abs_min=("abs_diff_min", "median"),
).reset_index())

In [ ]:
import seaborn as sns

if diff_long.empty:
    print("no matched pairs - nothing to plot")
else:
    SYSTEM_ORDER = ["TTA", "chained", "raw notes"]
    events_order = sorted(diff_long["event"].unique())

    fig, axes = plt.subplots(2, 2, figsize=(12, 9))
    panels = [
        (axes[0][0], "diff_min", "event", "signed, per event"),
        (axes[0][1], "diff_min", "system", "signed, pooled"),
        (axes[1][0], "abs_diff_min", "event", "absolute, per event"),
        (axes[1][1], "abs_diff_min", "system", "absolute, pooled"),
    ]
    for idx, (ax, col, x, title) in enumerate(panels):
        keys = [x] if x == "system" else [x, "system"]
        agg = (diff_long.assign(value=diff_long[col])
               .groupby(keys).agg(median=("value", "median"), n=("value", "size")).reset_index())
        present = [s for s in SYSTEM_ORDER if s in agg["system"].unique()]
        agg["system"] = pd.Categorical(agg["system"], categories=present, ordered=True)
        if x == "event":
            agg[x] = pd.Categorical(agg[x], categories=events_order, ordered=True)
        sns.barplot(data=agg, x=x, y="median", hue="system", hue_order=present,
                    errorbar=None, ax=ax, palette="muted")
        ax.axhline(0.0, color="k", lw=0.6)
        ax.set_title(title)
        ax.set_xlabel(x)
        ax.set_ylabel("median diff (min)" if col == "diff_min" else "median |diff| (min)")
        if x == "event":
            for label in ax.get_xticklabels():
                label.set_rotation(45)
                label.set_ha("right")
                label.set_fontsize(8)
        for ci, sysname in enumerate(present):
            if ci >= len(ax.containers):
                break
            sub = agg[agg["system"] == sysname].reset_index(drop=True)
            for bar, row in zip(ax.containers[ci].patches, sub.itertuples(index=False)):
                ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                        f"n={int(row.n)}", ha="center", va="bottom", fontsize=7)
        if idx == 0:
            ax.legend(loc="upper left", bbox_to_anchor=(0.0, 1.0), frameon=False)
        elif ax.get_legend_handles_labels()[0]:
            ax.legend().remove()
    fig.suptitle("Time difference vs. the structured reference (median of system - structured, matched pairs)", y=1.0)
    plt.tight_layout()
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(OUT_DIR / "tta_time_difference.png", dpi=150, bbox_inches="tight")
    print("saved", OUT_DIR / "tta_time_difference.png")